# Libraries and auxiliary functions

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from teleutils import preprocessing

In [ ]:
pd.options.mode.copy_on_write = True

# Data loading and processing

In [ ]:
%%time
df_numbers = pd.read_parquet('../assets/sample_numbers.parquet')
df_numbers

In [ ]:
%%time
df_numbers_a = pd.DataFrame(df_numbers.na).sample(100)
df_numbers_a[['na_clean','na_valid']] = df_numbers_a.apply(lambda row: pd.Series(preprocessing.normalize_number(row['na'])),axis=1)
df_numbers_a

In [ ]:
%%time
df_numbers_a_b = df_numbers
df_numbers_a_b[['na_clean','na_valid','nb_clean','nb_valid']] = df_numbers_a_b.apply(lambda row: pd.Series(preprocessing.normalize_number_pair(row['na'],row['nb'])),axis=1)
columns_to_keep = ['na', 'na_clean','na_valid','nb', 'nb_clean','nb_valid']
df_numbers_a_b = df_numbers_a_b[columns_to_keep]

number_columns = ['na', 'na_clean', 'nb', 'nb_clean']
for col in number_columns:
    df_numbers_a_b[f'len_{col}'] = df_numbers_a_b[col].str.len()

df_numbers_a_b

## Visual analysis

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, sharey=True, figsize=(8,7))

columns_to_plot = ['len_na', 'len_na_clean', 'len_nb', 'len_nb_clean']
plot_title = {
    'len_na': "Original length of A", 
    'len_nb': "Original length of B",
    'len_na_clean': "Normalized length of A", 
    'len_nb_clean': "Normalized length of B"}

x_labels_ = {
    'len_na': "Length", 
    'len_nb': "Length",
    'len_na_clean': "Length", 
    'len_nb_clean': "Length"
}

colors = {
    'len_na': "#BF8D30",
    'len_na_clean': "#D9B70D",
    'len_nb': "#2D4B73",    
    'len_nb_clean': "#99B4BF"
}

bins=np.arange(0,18,dtype='int')
bins = [0,6,9,10,11,12,15,16,99]
bins_ = ['0-5', '6-8', '9', '10', '11', '12-14','15', '16 ou +']
axes = axes.flatten()

# Loop through each column and plot a histogram
for i, column in enumerate(columns_to_plot):
    
    # Add the histogram
    # df_numbers_a_b[column].hist(ax=axes[i], edgecolor='white', color=colors[column], bins=bins)

    values = df_numbers_a_b[column].astype('float')
    heights, bins = np.histogram(values,bins=bins)
    heights = heights / 1000
    # bins_ = [str(x-1) for x in bins[1:]]
    axes[i].bar(bins_,heights, color=colors[column])
    
    # Add title and axis label
    axes[i].set_title(plot_title[column]) 
    axes[i].set_xlabel('Length', fontsize=10) 
    axes[i].set_ylabel('Quantity (thousands)', fontsize=10) 
    # axes[i].set_xticks(bins)

    # Limpeza e ajustes conforme os princípios de Alberto Cairo
    axes[i].grid(False)  # Remove grade para reduzir ruído visual
    axes[i].spines['top'].set_visible(False)  # Remove bordas superiores
    axes[i].spines['right'].set_visible(False)  # Remove bordas direitas
    axes[i].tick_params(axis='both', labelsize=8)  # Ticks menores e discretos

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()